# Week 3 Task: Advanced Data Analysis and Visualization in Logistics

**Project:** Exploratory analysis of hypothetical logistics operations  
**Tools:** Python, pandas, NumPy, Matplotlib  
**Note:** This is simulated data created for learning. It does not represent a real logistics company.

## Aim
In this project, I explore delivery time, shipment volume, distance, transportation cost, transport mode, and on-time delivery. I use summary statistics and charts to look for patterns and suggest practical improvements.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a repeatable hypothetical dataset
rng = np.random.default_rng(42)
n = 300

regions = np.array(["North", "South", "East", "West", "Central"])
modes = np.array(["Road", "Rail", "Air"])
priorities = np.array(["Standard", "Express"])

region = rng.choice(regions, n, p=[.25, .22, .16, .22, .15])
mode = rng.choice(modes, n, p=[.62, .25, .13])
priority = rng.choice(priorities, n, p=[.72, .28])
volume = rng.integers(20, 181, n)
distance = rng.integers(50, 1501, n)

base_days = distance / np.where(mode == "Air", 900, np.where(mode == "Rail", 450, 300))
regional_delay = np.select(
    [region == "North", region == "South", region == "East", region == "West"],
    [.3, 0, .6, .2], default=.4
)
priority_adjust = np.where(priority == "Express", -.7, 0)
delivery_days = np.maximum(
    .5, base_days + rng.normal(1.5, 1.1, n) + regional_delay + priority_adjust
)

mode_cost = np.select([mode == "Air", mode == "Rail", mode == "Road"], [3.8, 1.25, 1.8])
cost = (distance * mode_cost + volume * rng.uniform(3, 8, n) + rng.normal(0, 120, n)).clip(100)
delay_probability = np.clip(
    .08 + (delivery_days > 5) * .18 + (distance > 1000) * .1 + (mode == "Road") * .07,
    .03, .65
)
on_time = np.where(rng.random(n) < delay_probability, "No", "Yes")

df = pd.DataFrame({
    "Shipment_ID": [f"SH{1001+i}" for i in range(n)],
    "Region": region,
    "Transport_Mode": mode,
    "Priority": priority,
    "Shipment_Volume_Units": volume,
    "Distance_km": distance,
    "Delivery_Time_Days": np.round(delivery_days, 2),
    "Transport_Cost_USD": np.round(cost, 2),
    "On_Time_Delivery": on_time
})

df.head()


## 1. Understand the data
I use one row for each shipment. My dataset has 300 shipments and nine columns. I simulated the data so I can repeat the analysis without downloading a file.


In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values in each column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 2. Exploratory data analysis (EDA)
I calculate the mean, median, spread, and range of the numeric fields. The mean is the average, while the median is the middle value after sorting.


In [ ]:
numeric_cols = [
    "Shipment_Volume_Units", "Distance_km",
    "Delivery_Time_Days", "Transport_Cost_USD"
]
print(df[numeric_cols].describe().round(2))
print("\nMedian values:")
print(df[numeric_cols].median().round(2))


## 3. Delivery-time distribution
I use a histogram to see where delivery times are concentrated and whether unusually long deliveries appear.


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["Delivery_Time_Days"], bins=18, edgecolor="black")
plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery time (days)")
plt.ylabel("Number of shipments")
plt.tight_layout()
plt.show()


## 4. Distance and cost
I use a scatter plot to check whether longer routes tend to cost more. A relationship does not prove that distance is the only cause of cost.


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["Distance_km"], df["Transport_Cost_USD"], alpha=0.65)
plt.title("Distance and Transportation Cost")
plt.xlabel("Distance (km)")
plt.ylabel("Transportation cost (USD)")
plt.tight_layout()
plt.show()


## 5. Compare transport modes
I use a bar chart to compare the average delivery time for Road, Rail, and Air shipments.


In [ ]:
mode_summary = df.groupby("Transport_Mode").agg(
    Shipments=("Shipment_ID", "count"),
    Avg_Delivery_Days=("Delivery_Time_Days", "mean"),
    Avg_Cost_USD=("Transport_Cost_USD", "mean"),
    On_Time_Rate=("On_Time_Delivery", lambda x: (x == "Yes").mean() * 100)
).round(2)
print(mode_summary)

plt.figure(figsize=(7, 4))
plt.bar(mode_summary.index, mode_summary["Avg_Delivery_Days"])
plt.title("Average Delivery Time by Transport Mode")
plt.xlabel("Transport mode")
plt.ylabel("Average delivery time (days)")
plt.tight_layout()
plt.show()


## 6. Compare regions
I calculate the on-time rate as the percentage of shipments marked “Yes”. I use a bar chart to spot regions that may need closer review.


In [ ]:
region_summary = df.groupby("Region").agg(
    Shipments=("Shipment_ID", "count"),
    Avg_Delivery_Days=("Delivery_Time_Days", "mean"),
    Avg_Cost_USD=("Transport_Cost_USD", "mean"),
    On_Time_Rate=("On_Time_Delivery", lambda x: (x == "Yes").mean() * 100)
).round(2)
print(region_summary)

plt.figure(figsize=(7, 4))
plt.bar(region_summary.index, region_summary["On_Time_Rate"])
plt.title("On-Time Delivery Rate by Region")
plt.xlabel("Region")
plt.ylabel("On-time delivery (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


## 7. Correlation
I use correlation to check linear relationships. Values near +1 show a positive relationship, while values near -1 show a negative relationship. Correlation alone does not prove cause and effect.


In [ ]:
corr = df[numeric_cols].corr()
print(corr.round(2))

plt.figure(figsize=(7, 5))
plt.imshow(corr, vmin=-1, vmax=1, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=35, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlation")
plt.title("Correlation Between Numeric Variables")
plt.tight_layout()
plt.show()


## 8. Key findings and recommendations
I use the printed tables and charts to review the results from this run. I compare cost and delivery time together rather than choosing a transport mode based on speed alone. I would investigate regions with lower on-time rates, review long-distance shipments, and monitor cost per shipment. These results are only a starting point because my data is hypothetical.

## Conclusion
In this project, I simulated logistics data, checked its structure, calculated descriptive statistics, created visualizations, compared groups, and turned the results into recommendations. With real company data, I would validate these patterns before suggesting operational changes.
